In [ ]:
import sys
sys.path.append("../")

from AIRobot import AIRobot

#连接密钥：参照数字冰壶服务器界面中给出的连接信息填写，注意这个参数每次新启动服务器都会改变。
key = "tewisfdlws_b73e5f80-45a4-4b0d-9fbc-d76e4e6579a2"

#初始化AIRobot对象
myrobot = AIRobot(key, name="JupyterAI", host="curling-server-7788.jupyterhub.svc.cluster.local", port=7788)

#AIRobot对象开始接收并处理消息
myrobot.recv_forever()

In [ ]:
#=================================第三课测评任务答案=================================#
import socket
import time

#根据数字冰壶服务器界面中给出的连接信息修改CONNECTKEY，注意这个数据每次启动都会改变。
key = "olweshkgdt_d2e03453-3ff7-496a-bd50-14925f184fad"
#服务器IP
host = 'curling-server-7788.jupyterhub.svc.cluster.local'
#连接端口
port = 7788

#新建Socket对象
ai_sock =socket.socket()
#创建Socket连接
ai_sock.connect((host,port))
print("已建立socket连接", host, port)

#通过socket对象发送消息
def send_msg(sock, msg):
    print("  >>>> " + msg)
    #将消息数据从字符串类型转换为bytes类型后发送
    sock.send(msg.strip().encode())

#通过socket对象接收消息并进行解析
def recv_msg(sock):
    #为避免TCP粘包问题，数字冰壶服务器发送给AI选手的每一条信息均以0（数值为0的字节）结尾
    #这里采用了逐个字节接收后拼接的方式处理信息，多条信息之间以0为信息终结符
    buffer = bytearray()
    while True:
        #接收1个字节
        data = sock.recv(1)
        #接收到空数据或者信息处终结符(0)即中断循环
        if not data or data == b'\0':
            time.sleep(0.1)
            break
        #将当前字节拼接到缓存中
        buffer.extend(data)
    #将消息数据从bytes类型转换为字符串类型后去除前后空格
    msg_str = buffer.decode().strip()
    print("<<<< " + msg_str)

    #用空格将消息字符串分隔为列表
    msg_list = msg_str.split(" ")
    #列表中第一个项为消息代码
    msg_code = msg_list[0]
    #列表中后续的项为各个参数
    msg_list.pop(0)
    #返回消息代码和消息参数列表
    return msg_code, msg_list

#通过socket对象发送连接密钥
send_msg(ai_sock, "CONNECTKEY:" + key)
#初始化先手壶和后手壶的坐标列表
init_x, init_y, gote_x, gote_y = [0]*8, [0]*8, [0]*8, [0]*8

while(1):
    #接收消息并解析
    msg_code, msg_list = recv_msg(ai_sock)
    #如果消息代码是"ISREADY"
    if msg_code == "ISREADY":
        #发送"READYOK"
        send_msg(ai_sock, "READYOK")
        #发送实验模式（正式比赛时设定为0）
        send_msg(ai_sock, "EXPMODE 2")
        time.sleep(0.5)
        #发送"NAME"和AI选手名
        send_msg(ai_sock, "NAME JupyterAI")
        break

#########################请在下方补全代码实现指定功能######################

P = [[2.375, 4.111],[2.375, 4.88],[1.606, 4.88],[3.144, 4.88],
     [2.375, 5.649],[1.1552, 6.0998],[3.5948, 6.0998],[2.375, 6.605]]
sum_diff_x = 0
sum_diff_y = 0

for m in range(8):
    while(1):
        #接收消息并解析
        msg_code, msg_list = recv_msg(ai_sock)
        #如果消息代码是"GO"
        if msg_code == "GO":
            print("============先手方第" + str(m+1) + "壶============")
            v0 = 3.6 - 0.123*P[m][1]
            h0 = P[m][0] - 2.375
            #发送投壶消息：初速度为v0；横向偏移为h0，初始角速度为0
            send_msg(ai_sock, "BESTSHOT " + str(v0) + " " + str(h0) + " 0")
        if msg_code=="POSITION":
            
            for n in range(8):
                init_x[n], init_y[n] = float(msg_list[n*4]), float(msg_list[n*4+1])
                gote_x[n], gote_y[n] = float(msg_list[n*4+2]), float(msg_list[n*4+3])
            if (init_x[m]>0.0001) and (init_y[m]>0.0001):
                diff_x = init_x[m]-P[m][0]
                diff_y = init_y[m]-P[m][1]
                print("第" + str(m+1) + "壶相比预设坐标偏差为(%.4f, %.4f)" % (diff_x, diff_y))
                sum_diff_x = sum_diff_x + diff_x
                sum_diff_y = sum_diff_y + diff_y
                break

print("平均坐标偏差为(%.4f, %.4f)" % (sum_diff_x/8.0, sum_diff_y/8.0))